# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShayanAhmed12/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Lane (confirmed for lock-in): CTR / Engagement Opportunity Scoring.**

**The rule in plain words:** *"A visible page that earned clearly fewer clicks than pages at a similar position over the last two weeks, and has a lot of impressions, is worth a snippet review first. The more clicks it is missing, the higher it goes."*

- **Score:** `clicks_at_stake = (expected_ctr − ctr_f) × impr_f`, where `expected_ctr` is the median feature-window CTR of pages in the same position bucket (1–3, 4–10, 11–20). Only pages with ≥500 feature-window impressions and CTR below **half** their bucket median get a score. Everyone else scores 0.
- **Reason code (one):** `low_ctr_for_position`
- **Action label:** `review_title_meta` (rewrite the title tag and meta description, check search-intent match)

**Data:** the same lane frame as ML-04: warehouse `month=2026-03`, decision moment **2026-03-16**. Features use 1–15 March only; the outcome (`under_captures_next`, 16–31 March) is used **only to check** the signals and score the queue, never as an input.

**Before encoding it, two signals the rule leans on get checked** (bucket tables with *n*):
1. **CTR-vs-position gap** (the signal behind FlyRank's CTR-fix logic): do pages that under-click for their position *keep* under-clicking?
2. **Volume** (the signal behind quick-win logic): is the under-clicking signal more reliable on higher-impression pages?


In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, json, duckdb, numpy as np, pandas as pd
from huggingface_hub import hf_hub_download, get_token

# Repo root (works from work/notebooks/ locally; Colab clones the repo)
import sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
else:
    while not os.path.isdir("work") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")

assert get_token(), "No Hugging Face token: log in (hf auth login) or add HF_TOKEN in Colab Secrets"
REPO = "FlyRank/internship-warehouse"
con = duckdb.connect()
for name, path in {"fact_mar": "fact_content_daily_performance/month=2026-03/data_0.parquet",
                   "dim_content": "dim_content.parquet"}.items():
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{hf_hub_download(REPO, path, repo_type='dataset')}')")

DECISION = "2026-03-16"
frame = con.sql(f"""
    WITH f AS (SELECT client_hash_id, content_hash_id,
                      SUM(gsc_impressions) AS impr_f, SUM(gsc_clicks) AS clicks_f,
                      SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS pos_f
               FROM fact_mar WHERE gsc_data_available IS TRUE AND report_date <  DATE '{DECISION}' GROUP BY 1, 2),
         o AS (SELECT content_hash_id, SUM(gsc_impressions) AS impr_o, SUM(gsc_clicks) AS clicks_o
               FROM fact_mar WHERE gsc_data_available IS TRUE AND report_date >= DATE '{DECISION}' GROUP BY 1)
    SELECT f.client_hash_id, f.content_hash_id, f.impr_f, f.clicks_f,
           100.0 * f.clicks_f / f.impr_f AS ctr_f, f.pos_f, c.content_type, c.main_intent,
           DATE_DIFF('day', c.content_created_date, DATE '{DECISION}') AS age_days,
           o.impr_o, 100.0 * o.clicks_o / o.impr_o AS ctr_o
    FROM f JOIN o USING (content_hash_id) LEFT JOIN dim_content c USING (content_hash_id)
    WHERE f.impr_f >= 100 AND o.impr_o >= 100 AND f.pos_f BETWEEN 1 AND 20
""").df()
frame["pos_bucket"] = pd.cut(frame["pos_f"], [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"])
# Label (outcome window) - for CHECKING only
frame["under_captures_next"] = (frame["ctr_o"] < 0.5 * frame.groupby("pos_bucket", observed=True)["ctr_o"].transform("median")).astype(int)
# Feature-window expectation - what the rule is allowed to use
frame["expected_ctr"] = frame.groupby("pos_bucket", observed=True)["ctr_f"].transform("median")
frame["ctr_ratio_f"] = frame["ctr_f"] / frame["expected_ctr"]
base_rate = frame["under_captures_next"].mean()
print(f"Lane frame: {len(frame):,} pages, {frame['client_hash_id'].nunique()} clients | base rate {base_rate:.3f}")


Lane frame: 57,392 pages, 36 clients | base rate 0.398


### Signal check 1: CTR-vs-position gap (flag-linked: CTR-fix logic)

Bucket pages by how their **feature-window** CTR compares to their position bucket's median, then look at how often they under-capture in the **next** two weeks.

In [2]:
bins = [-0.01, 0.25, 0.5, 0.75, 1.0, 1.5, np.inf]
labels = ["<25% of exp", "25-50%", "50-75%", "75-100%", "100-150%", ">150%"]
frame["gap_bucket"] = pd.cut(frame["ctr_ratio_f"], bins, labels=labels)
sig1 = frame.groupby("gap_bucket", observed=True)["under_captures_next"].agg(n="size", rate="mean")
sig1["lift_vs_base"] = sig1["rate"] / base_rate
print(f"Base rate: {base_rate:.3f}")
print(sig1.round(3).to_string())


Base rate: 0.398
                 n   rate  lift_vs_base
gap_bucket                             
<25% of exp  19457  0.672         1.687
25-50%        2624  0.480         1.205
50-75%        3214  0.380         0.953
75-100%       3406  0.312         0.784
100-150%      5557  0.274         0.687
>150%        23134  0.204         0.512


### Signal check 2: volume (flag-linked: quick-win logic)

Among pages that under-click in the feature window (CTR < 50% of expected), does the signal get more reliable as impressions grow? If small pages are mostly noise, the rule needs a volume floor.

In [3]:
low = frame[frame["ctr_ratio_f"] < 0.5].copy()
low["volume_bucket"] = pd.cut(low["impr_f"], [99, 250, 500, 1000, 5000, np.inf],
                              labels=["100-249", "250-499", "500-999", "1k-5k", "5k+"])
sig2 = low.groupby("volume_bucket", observed=True)["under_captures_next"].agg(n="size", rate="mean")
sig2["lift_vs_base"] = sig2["rate"] / base_rate
print("Pages under-clicking in the feature window, by feature-window impressions:")
print(sig2.round(3).to_string())


Pages under-clicking in the feature window, by feature-window impressions:
                  n   rate  lift_vs_base
volume_bucket                           
100-249        8048  0.736         1.848
250-499        5707  0.653         1.639
500-999        3458  0.574         1.441
1k-5k          4137  0.540         1.356
5k+             730  0.640         1.607


### Verdicts

**Signal 1: CTR-vs-position gap → CONFIRMED.** The relationship is monotonic and strong. Pages under 25% of their bucket's expected CTR under-captured again in the next two weeks **67%** of the time (n = 19,457), against a **40%** base rate (1.7× lift). Pages above 150% of expected did so only 20% of the time (n = 23,134). Under-clicking for your position is persistent, so the rule can lean on it.

**Signal 2: volume → MIXED.** I expected the signal to get *more* reliable with more impressions. It doesn't: among under-clicking pages, the smallest bucket (100–249 impressions) repeats most often (74%, n = 8,048), the middle buckets drop to 54–57%, and 5k+ rises again (64%, n = 730). So volume does **not** buy reliability here; small pages under-click persistently too. That changes *why* the rule has a volume floor: the ≥500-impression floor is about **impact** (clicks at stake worth an editor's hour), not about trusting the signal more. A negative result that saved the rule from a wrong justification.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
VOLUME_FLOOR, GAP_CUTOFF = 500, 0.5
eligible = (frame["impr_f"] >= VOLUME_FLOOR) & (frame["ctr_ratio_f"] < GAP_CUTOFF)
frame["baseline_score"] = np.where(eligible, (frame["expected_ctr"] - frame["ctr_f"]) / 100 * frame["impr_f"], 0.0)
frame["reason_code"] = np.where(eligible, "low_ctr_for_position", "")
frame["action"] = np.where(eligible, "review_title_meta", "no_action")

queue = (frame.sort_values("baseline_score", ascending=False)
              .reset_index(drop=True)
              .assign(rank=lambda d: d.index + 1))
out_cols = ["rank", "content_hash_id", "client_hash_id", "content_type", "main_intent", "pos_bucket",
            "pos_f", "impr_f", "clicks_f", "ctr_f", "expected_ctr", "baseline_score", "reason_code", "action"]
os.makedirs("work/outputs", exist_ok=True)
queue[out_cols].round(4).to_csv("work/outputs/baseline_action_score.csv", index=False)   # stays out of git (CI guard)
print(f"Wrote work/outputs/baseline_action_score.csv: {len(queue):,} rows, {int(eligible.sum()):,} flagged review_title_meta")

# How good is the queue? Scored against the OUTCOME window it never saw.
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(labels)[order[:k]].mean()
y = frame["under_captures_next"].values
naive = np.where(frame["impr_f"] >= VOLUME_FLOOR, -frame["ctr_f"], -np.inf)   # 'lowest raw CTR first' strawman
metrics = {"slice": "warehouse month=2026-03, decision 2026-03-16, features 1-15 Mar, outcome 16-31 Mar",
           "pages": int(len(frame)), "clients": int(frame["client_hash_id"].nunique()),
           "base_rate": round(float(base_rate), 4), "flagged": int(eligible.sum())}
for k in (10, 50, 100, 500):
    metrics[f"baseline_p@{k}"] = round(float(precision_at_k(frame["baseline_score"], y, k)), 4)
    metrics[f"naive_low_ctr_p@{k}"] = round(float(precision_at_k(naive, y, k)), 4)
with open("work/outputs/w04_baseline_metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)
print(json.dumps(metrics, indent=2))


Wrote work/outputs/baseline_action_score.csv: 57,392 rows, 8,347 flagged review_title_meta
{
  "slice": "warehouse month=2026-03, decision 2026-03-16, features 1-15 Mar, outcome 16-31 Mar",
  "pages": 57392,
  "clients": 36,
  "base_rate": 0.3981,
  "flagged": 8347,
  "baseline_p@10": 0.9,
  "naive_low_ctr_p@10": 0.6,
  "baseline_p@50": 0.72,
  "naive_low_ctr_p@50": 0.74,
  "baseline_p@100": 0.78,
  "naive_low_ctr_p@100": 0.77,
  "baseline_p@500": 0.708,
  "naive_low_ctr_p@500": 0.634
}


## 3. Top-10 review

For each of the top ten: the action, why it's there (the numbers behind the score), and what would make it wrong. The lines are built from each row's own numbers. I read them and summarise the weak picks in section 4.

In [5]:
def wrong_if(r):
    risks = []
    if r["clicks_f"] == 0:
        risks.append("0 clicks on big impressions smells like a tracking/SERP-feature issue (image/answer box), not a bad snippet")
    if r["pos_bucket"] == "1-3":
        risks.append("at positions 1-3 an answer box or a strong brand result above it can absorb the clicks")
    if r["pos_bucket"] == "11-20":
        risks.append("on page 2 the fix is ranking, not the snippet; few people see it at all")
    if str(r["main_intent"]).lower() in ("informational", "nan", "none"):
        risks.append("informational queries are often answered in the SERP itself (zero-click searches)")
    if r["impr_f"] > 20 * frame["impr_f"].median():
        risks.append("huge impressions may come from one broad, low-intent query where low CTR is normal")
    return "; ".join(risks[:2]) or "a one-off dip: two weeks may simply be a quiet fortnight"

top10 = queue.head(10).copy()
for _, r in top10.iterrows():
    print(f"#{r['rank']:>2} {r['action']} [{r['reason_code']}] | {r['content_type']}, {r['main_intent']}, pos {r['pos_f']:.1f}")
    print(f"     why: {int(r['impr_f']):,} impressions, CTR {r['ctr_f']:.2f}% vs {r['expected_ctr']:.2f}% expected "
          f"-> ~{r['baseline_score']:.0f} clicks missing in 2 weeks")
    print(f"     wrong if: {wrong_if(r)}")


# 1 review_title_meta [low_ctr_for_position] | keyword article, informational, pos 2.9
     why: 73,639 impressions, CTR 0.02% vs 0.27% expected -> ~179 clicks missing in 2 weeks
     wrong if: at positions 1-3 an answer box or a strong brand result above it can absorb the clicks; informational queries are often answered in the SERP itself (zero-click searches)
# 2 review_title_meta [low_ctr_for_position] | keyword article, commercial, pos 6.0
     why: 86,860 impressions, CTR 0.06% vs 0.20% expected -> ~125 clicks missing in 2 weeks
     wrong if: huge impressions may come from one broad, low-intent query where low CTR is normal
# 3 review_title_meta [low_ctr_for_position] | keyword article, commercial, pos 4.8
     why: 58,553 impressions, CTR 0.00% vs 0.20% expected -> ~118 clicks missing in 2 weeks
     wrong if: huge impressions may come from one broad, low-intent query where low CTR is normal
# 4 review_title_meta [low_ctr_for_position] | keyword article, transactional, pos 8.4
 

## 4. Weak picks + leakage check

**How the queue did (scored against the outcome window it never saw):** at the top 10 it was right **9/10** (naive "lowest CTR" rule: 6/10). At top 500 it scored **0.708** vs **0.634**. But at **top 50 it tied the naive rule (0.72 vs 0.74)**, and at top 100 too (0.78 vs 0.77). So the position adjustment helps at the very top and deep in the list, not across the board. That's the honest bar for Week 5: a model must beat **0.72 at P@50** (base rate 0.40) on held-out clients.

**Weak picks I see in the top 10:**
- **#1 and #9 sit at positions 1–3 on informational queries.** These are the likeliest false alarms: answer boxes or featured snippets above a top result can take the clicks, and no title rewrite fixes that.
- **Near-zero CTR on very large impressions (#3, #4: CTR ≈ 0.00% on ~50–60k impressions)** looks less like "a weak snippet" and more like the page ranking for one broad query it doesn't really answer. The action might be intent-matching or de-optimising, not a better title.
- **Concentration:** the top 10 comes from only 5 clients, and one client holds 38% of the top 50. The rule may partly be ranking *sites* (their template, their niche), not pages.
- **One expectation for all content:** all ten are keyword articles. The tier medians are pooled across content types and intents, so the "expected CTR" for an informational page is probably set too high.

**Leakage:** the rule uses only `impr_f`, `ctr_f`, `pos_f` and the bucket median of `ctr_f`, all from 1–15 March. The outcome window and the label appear only when *scoring* the queue. The release ships no product flags. Checked in the cell below.

In [6]:
# Weak-pick evidence and leakage checks
top10_ids = set(top10["content_hash_id"])
chk = frame[frame["content_hash_id"].isin(top10_ids)]
print(f"Top-10 that actually under-captured next window (check only): {int(chk['under_captures_next'].sum())}/10")
print(f"Top-10 with zero clicks in the feature window: {int((top10['clicks_f'] == 0).sum())}")
print(f"Top-10 clients represented: {top10['client_hash_id'].nunique()} | largest client share of top 50: "
      f"{queue.head(50)['client_hash_id'].value_counts(normalize=True).iloc[0]:.0%}")

# Leakage: the rule's inputs must all come from the feature window
RULE_INPUTS = ["impr_f", "ctr_f", "pos_f", "expected_ctr"]
assert not any(c.endswith("_o") for c in RULE_INPUTS), "outcome-window column in the rule"
assert "under_captures_next" not in RULE_INPUTS
print("Rule inputs:", RULE_INPUTS, "-> all from 1-15 March; no product flags exist in the release; label used only for scoring.")


Top-10 that actually under-captured next window (check only): 9/10
Top-10 with zero clicks in the feature window: 0
Top-10 clients represented: 5 | largest client share of top 50: 38%
Rule inputs: ['impr_f', 'ctr_f', 'pos_f', 'expected_ctr'] -> all from 1-15 March; no product flags exist in the release; label used only for scoring.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.